#  Lab 12: Text Vectorization — TF-IDF & Word Embeddings

---

##  Aim
To apply text vectorization techniques — Bag of Words, TF-IDF, and Word Embeddings — for document similarity computation, keyword extraction, and semantic analysis tasks.

##  Theory

### Why Vectorization?
Machine learning models work with **numbers, not text**. Vectorization is the process of converting text into numerical representations that capture meaning. The choice of vectorization technique directly impacts model performance.

### Evolution of Text Representation

```
One-Hot Encoding       →   Bag of Words (BoW)   →   TF-IDF
(sparse, no meaning)       (word counts)             (weighted counts)
          │
          ▼
Word2Vec / GloVe       →   Sentence Embeddings  →   Contextual Embeddings
(dense, semantic)          (sentence-level)          (BERT, 2024+)
```

### Key Concepts
| Technique | Type | Captures Meaning? | Size |
|---|---|---|---|
| **One-Hot** | Sparse |  No | vocab size |
| **Bag of Words** | Sparse |  Word counts only | vocab size |
| **TF-IDF** | Sparse |  Importance, not semantics | vocab size |
| **Word2Vec** | Dense |  Semantic similarity | 100–300 dims |
| **GloVe** | Dense |  Global co-occurrence | 50–300 dims |
| **Sentence-BERT** | Dense |  Full sentence semantics | 384–768 dims |

### TF-IDF Formula
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

$$\text{IDF}(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1$$

Where: `t` = term, `d` = document, `N` = total documents, `df(t)` = documents containing term `t`

---

##  Part 1: Setup

In [1]:
!pip install scikit-learn gensim sentence-transformers nltk numpy matplotlib seaborn -q

In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import normalize

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [3]:
# ─── Working Corpus ──────────────────────────────────────────────
# A small but representative document collection covering tech topics
corpus = [
    "Machine learning is a subset of artificial intelligence focused on learning from data.",
    "Deep learning uses neural networks with many layers to learn complex patterns in data.",
    "Natural language processing enables computers to understand and generate human language.",
    "Computer vision allows machines to interpret and analyze images and video content.",
    "Reinforcement learning trains agents to make decisions by rewarding correct behavior.",
    "Data science combines statistics, programming, and domain expertise to extract insights.",
    "Python is the most popular programming language used in machine learning and data science.",
    "Neural networks are inspired by the structure of neurons in the human brain.",
    "Artificial intelligence research focuses on building systems that can reason like humans.",
    "TF-IDF is a text vectorization method that weighs terms by their importance in a document."
]

doc_labels = [
    "ML Basics", "Deep Learning", "NLP", "Computer Vision", "RL",
    "Data Science", "Python/ML", "Neural Nets", "AI Research", "TF-IDF"
]

print(f"📄 Corpus loaded: {len(corpus)} documents")
for i, (label, doc) in enumerate(zip(doc_labels, corpus)):
    print(f"  [{i}] {label:<16}: {doc[:60]}...")

📄 Corpus loaded: 10 documents
  [0] ML Basics       : Machine learning is a subset of artificial intelligence focu...
  [1] Deep Learning   : Deep learning uses neural networks with many layers to learn...
  [2] NLP             : Natural language processing enables computers to understand ...
  [3] Computer Vision : Computer vision allows machines to interpret and analyze ima...
  [4] RL              : Reinforcement learning trains agents to make decisions by re...
  [5] Data Science    : Data science combines statistics, programming, and domain ex...
  [6] Python/ML       : Python is the most popular programming language used in mach...
  [7] Neural Nets     : Neural networks are inspired by the structure of neurons in ...
  [8] AI Research     : Artificial intelligence research focuses on building systems...
  [9] TF-IDF          : TF-IDF is a text vectorization method that weighs terms by t...


---
##  Part 2: Bag of Words (BoW)

**Bag of Words** is the simplest vectorization method. It represents each document as a **vector of word counts**, ignoring grammar and word order. The result is a sparse matrix of shape `(n_documents × vocabulary_size)`.

In [4]:
# Build Bag-of-Words matrix
bow_vectorizer = CountVectorizer(
    stop_words='english',   # remove English stopwords automatically
    lowercase=True,
    min_df=1                # include terms appearing in at least 1 doc
)

bow_matrix = bow_vectorizer.fit_transform(corpus)
vocabulary = bow_vectorizer.get_feature_names_out()

print(f"📊 BoW Matrix Shape : {bow_matrix.shape}")
print(f"   (rows=documents, columns=unique vocab terms)")
print(f"\n📚 Vocabulary Size  : {len(vocabulary)} terms")
print(f"\n🔤 First 20 vocab terms:")
print(list(vocabulary[:20]))

📊 BoW Matrix Shape : (10, 71)
   (rows=documents, columns=unique vocab terms)

📚 Vocabulary Size  : 71 terms

🔤 First 20 vocab terms:
['agents', 'allows', 'analyze', 'artificial', 'behavior', 'brain', 'building', 'combines', 'complex', 'computer', 'computers', 'content', 'correct', 'data', 'decisions', 'deep', 'document', 'domain', 'enables', 'expertise']


In [5]:
import pandas as pd

# View BoW matrix as a readable DataFrame (first 15 columns)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    index=doc_labels,
    columns=vocabulary
)

# Show only columns where at least one doc has count > 0
# and select 15 interesting columns
interesting_cols = ['learning', 'data', 'neural', 'networks', 'language',
                    'intelligence', 'machine', 'deep', 'human', 'science',
                    'python', 'image', 'decision', 'text', 'computer']
available = [c for c in interesting_cols if c in bow_df.columns]

print("📊 BoW Matrix (selected columns):")
print(bow_df[available].to_string())
print("\n📌 Most cells are 0 — BoW matrices are very SPARSE.")

📊 BoW Matrix (selected columns):
                 learning  data  neural  networks  language  intelligence  machine  deep  human  science  python  text  computer
ML Basics               2     1       0         0         0             1        1     0      0        0       0     0         0
Deep Learning           1     1       1         1         0             0        0     1      0        0       0     0         0
NLP                     0     0       0         0         2             0        0     0      1        0       0     0         0
Computer Vision         0     0       0         0         0             0        0     0      0        0       0     0         1
RL                      1     0       0         0         0             0        0     0      0        0       0     0         0
Data Science            0     1       0         0         0             0        0     0      0        1       0     0         0
Python/ML               1     1       0         0         1     

In [6]:
# Sparsity calculation
total_cells   = bow_matrix.shape[0] * bow_matrix.shape[1]
nonzero_cells = bow_matrix.nnz
sparsity      = 1 - (nonzero_cells / total_cells)

print(f"📊 Sparsity Statistics:")
print(f"   Total cells     : {total_cells}")
print(f"   Non-zero cells  : {nonzero_cells}")
print(f"   Zero cells      : {total_cells - nonzero_cells}")
print(f"   Sparsity        : {sparsity:.2%}")
print("\n📌 This is why dense embeddings were invented — BoW wastes memory at scale.")

📊 Sparsity Statistics:
   Total cells     : 710
   Non-zero cells  : 86
   Zero cells      : 624
   Sparsity        : 87.89%

📌 This is why dense embeddings were invented — BoW wastes memory at scale.


###  Limitation of BoW
Consider two sentences:
- `"The dog bit the man"` → `{dog:1, bit:1, man:1}`
- `"The man bit the dog"` → `{dog:1, bit:1, man:1}`

**Both produce the exact same vector!** BoW completely **ignores word order**, which means it can't distinguish meaning differences caused by word arrangement.

---
##  Part 3: TF-IDF Vectorization

**TF-IDF** improves on BoW by down-weighting words that appear in *many* documents (common words carry less information) and up-weighting words that appear in *few* documents (rare words are more distinctive).

In [7]:
# Build TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    max_features=500,       # keep top 500 terms
    ngram_range=(1, 2),     # include unigrams AND bigrams
    sublinear_tf=True       # dampen high frequency terms (log TF)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
tfidf_vocab  = tfidf_vectorizer.get_feature_names_out()

print(f"📊 TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"   (includes unigrams + bigrams, max 500 features)")

📊 TF-IDF Matrix Shape: (10, 144)
   (includes unigrams + bigrams, max 500 features)


In [8]:
# Build TF-IDF DataFrame for inspection
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=doc_labels,
    columns=tfidf_vocab
)

# Show top-5 highest TF-IDF scoring terms per document
print("📌 Top-5 TF-IDF Keywords Per Document:\n")
print(f"{'Document':<18} | Top Keywords (TF-IDF score)")
print("-" * 80)
for label, row in tfidf_df.iterrows():
    top5 = row.nlargest(5)
    keywords = ', '.join([f"{term}({score:.3f})" for term, score in top5.items()])
    print(f"{label:<18} | {keywords}")

📌 Top-5 TF-IDF Keywords Per Document:

Document           | Top Keywords (TF-IDF score)
--------------------------------------------------------------------------------
ML Basics          | learning(0.323), focused(0.288), focused learning(0.288), intelligence focused(0.288), learning subset(0.288)
Deep Learning      | complex(0.242), complex patterns(0.242), deep(0.242), deep learning(0.242), layers(0.242)
NLP                | language(0.351), computers(0.244), computers understand(0.244), enables(0.244), enables computers(0.244)
Computer Vision    | allows(0.243), allows machines(0.243), analyze(0.243), analyze images(0.243), computer(0.243)
RL                 | agents(0.247), agents make(0.247), behavior(0.247), correct(0.247), correct behavior(0.247)
Data Science       | combines(0.253), combines statistics(0.253), domain(0.253), domain expertise(0.253), expertise(0.253)
Python/ML          | language used(0.268), popular(0.268), popular programming(0.268), programming language(0.26

In [9]:
# ─── Manual TF-IDF Calculation (to understand the math) ──────────
import math

def compute_tf(document, term):
    words = document.lower().split()
    return words.count(term) / len(words)

def compute_idf(corpus, term):
    N  = len(corpus)
    df = sum(1 for doc in corpus if term in doc.lower())
    return math.log((N + 1) / (df + 1)) + 1   # sklearn smooth formula

def compute_tfidf(corpus, term, doc_index):
    tf  = compute_tf(corpus[doc_index], term)
    idf = compute_idf(corpus, term)
    return tf * idf

# Demonstrate on key terms across all documents
demo_terms = ['learning', 'data', 'neural', 'language']

print("🔢 Manual TF-IDF Scores:\n")
for term in demo_terms:
    idf_val = compute_idf(corpus, term)
    print(f"Term: '{term}' | IDF = {idf_val:.4f}")
    for i, label in enumerate(doc_labels):
        tf_val  = compute_tf(corpus[i], term)
        tfidf   = tf_val * idf_val
        if tf_val > 0:
            print(f"   [{label}] TF={tf_val:.4f} × IDF={idf_val:.4f} = TF-IDF={tfidf:.4f}")
    print()

🔢 Manual TF-IDF Scores:

Term: 'learning' | IDF = 1.7885
   [ML Basics] TF=0.1538 × IDF=1.7885 = TF-IDF=0.2751
   [Deep Learning] TF=0.0714 × IDF=1.7885 = TF-IDF=0.1277
   [RL] TF=0.0909 × IDF=1.7885 = TF-IDF=0.1626
   [Python/ML] TF=0.0714 × IDF=1.7885 = TF-IDF=0.1277

Term: 'data' | IDF = 1.7885
   [Data Science] TF=0.0909 × IDF=1.7885 = TF-IDF=0.1626
   [Python/ML] TF=0.0714 × IDF=1.7885 = TF-IDF=0.1277

Term: 'neural' | IDF = 2.2993
   [Deep Learning] TF=0.0714 × IDF=2.2993 = TF-IDF=0.1642
   [Neural Nets] TF=0.0769 × IDF=2.2993 = TF-IDF=0.1769

Term: 'language' | IDF = 2.2993
   [NLP] TF=0.0909 × IDF=2.2993 = TF-IDF=0.2090
   [Python/ML] TF=0.0714 × IDF=2.2993 = TF-IDF=0.1642



In [10]:
# TF-IDF Heatmap — visualize the weight matrix
# Pick 12 most discriminative terms for display
top_terms_per_doc = set()
for label, row in tfidf_df.iterrows():
    top_terms_per_doc.update(row.nlargest(3).index.tolist())
top_terms_list = list(top_terms_per_doc)[:20]

fig, ax = plt.subplots(figsize=(14, 6))
heatmap_data = tfidf_df[top_terms_list].values
sns.heatmap(
    heatmap_data,
    ax=ax,
    xticklabels=top_terms_list,
    yticklabels=doc_labels,
    cmap='YlOrRd',
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'TF-IDF Score'}
)
ax.set_title("TF-IDF Weight Heatmap — Top Keywords per Document",
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('tfidf_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print("📌 Brighter = higher TF-IDF = more distinctive for that document")

📌 Brighter = higher TF-IDF = more distinctive for that document


---
##  Part 4: Document Similarity with TF-IDF

**Cosine Similarity** measures the angle between two vectors in high-dimensional space.  
A score of `1.0` = identical direction (same meaning), `0.0` = orthogonal (no shared content).

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

In [11]:
# Compute full cosine similarity matrix between all documents
cos_sim_matrix = cosine_similarity(tfidf_matrix)

sim_df = pd.DataFrame(cos_sim_matrix, index=doc_labels, columns=doc_labels)

# Plot similarity heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(len(doc_labels), dtype=bool)  # mask diagonal (self-similarity = 1.0)
sns.heatmap(
    sim_df,
    ax=ax,
    cmap='coolwarm',
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    vmin=0, vmax=1,
    square=True,
    cbar_kws={'label': 'Cosine Similarity'}
)
ax.set_title("Document Cosine Similarity Matrix (TF-IDF)",
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('cosine_similarity.png', dpi=120, bbox_inches='tight')
plt.show()

In [12]:
# Find the most similar pair of documents (excluding self-similarity)
sim_matrix_no_diag = cos_sim_matrix.copy()
np.fill_diagonal(sim_matrix_no_diag, 0)

max_idx = np.unravel_index(np.argmax(sim_matrix_no_diag), sim_matrix_no_diag.shape)
min_idx = np.unravel_index(np.argmin(sim_matrix_no_diag), sim_matrix_no_diag.shape)

print("🔝 Most Similar Documents:")
print(f"   '{doc_labels[max_idx[0]]}' ↔ '{doc_labels[max_idx[1]]}'")
print(f"   Similarity Score: {cos_sim_matrix[max_idx]:.4f}")
print(f"   Doc A: {corpus[max_idx[0]]}")
print(f"   Doc B: {corpus[max_idx[1]]}")

print("\n🔻 Least Similar Documents:")
print(f"   '{doc_labels[min_idx[0]]}' ↔ '{doc_labels[min_idx[1]]}'")
print(f"   Similarity Score: {cos_sim_matrix[min_idx]:.4f}")

🔝 Most Similar Documents:
   'ML Basics' ↔ 'Python/ML'
   Similarity Score: 0.2584
   Doc A: Machine learning is a subset of artificial intelligence focused on learning from data.
   Doc B: Python is the most popular programming language used in machine learning and data science.

🔻 Least Similar Documents:
   'ML Basics' ↔ 'ML Basics'
   Similarity Score: 1.0000


In [13]:
# ─── Search Engine: Find most relevant documents for a query ──────
def search_documents(query, vectorizer, doc_matrix, labels, corpus, top_k=3):
    """
    Given a query string, rank documents by cosine similarity to the query.
    This is exactly how basic search engines work!
    """
    query_vec   = vectorizer.transform([query])             # vectorize the query
    similarities = cosine_similarity(query_vec, doc_matrix).flatten()
    ranked_idx  = np.argsort(similarities)[::-1][:top_k]   # top-k descending
    
    print(f"🔍 Query: '{query}'")
    print(f"\n{'Rank':<5} | {'Document':<18} | {'Score':<8} | Content")
    print("-" * 85)
    for rank, idx in enumerate(ranked_idx, 1):
        print(f"{rank:<5} | {labels[idx]:<18} | {similarities[idx]:.4f}   | {corpus[idx][:50]}...")


# Test with different queries
queries = [
    "how do neural networks learn patterns",
    "programming language for AI development",
    "understanding images using computers"
]

for q in queries:
    print()
    search_documents(q, tfidf_vectorizer, tfidf_matrix, doc_labels, corpus)
    print()


🔍 Query: 'how do neural networks learn patterns'

Rank  | Document           | Score    | Content
-------------------------------------------------------------------------------------
1     | Deep Learning      | 0.4945   | Deep learning uses neural networks with many layer...
2     | Neural Nets        | 0.3080   | Neural networks are inspired by the structure of n...
3     | TF-IDF             | 0.0000   | TF-IDF is a text vectorization method that weighs ...


🔍 Query: 'programming language for AI development'

Rank  | Document           | Score    | Content
-------------------------------------------------------------------------------------
1     | Python/ML          | 0.4189   | Python is the most popular programming language us...
2     | NLP                | 0.1909   | Natural language processing enables computers to u...
3     | Data Science       | 0.1170   | Data science combines statistics, programming, and...


🔍 Query: 'understanding images using computers'

Rank  | Docu

---
##  Part 5: Word Embeddings with Word2Vec

**Word2Vec** (2013, Google) maps each word to a **dense vector** in a continuous space such that semantically similar words cluster together.

Two architectures:
- **CBOW** (Continuous Bag of Words) — predict target word from context words
- **Skip-Gram** — predict context words from the target word

Key property: **Word arithmetic works!**
```
king - man + woman ≈ queen
Paris - France + Italy ≈ Rome
```

In [15]:
# ════════════════════════════════════════════════════════════════════
# Word Embeddings (No Gensim — SVD Co-occurrence, scipy-safe)
# ════════════════════════════════════════════════════════════════════
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Extended corpus ──────────────────────────────────────────────────
extended_corpus = corpus + [
    "Supervised learning requires labeled training data for model training.",
    "Unsupervised learning discovers hidden patterns in unlabeled data.",
    "Gradient descent optimizes neural network weights during training.",
    "Convolutional neural networks are powerful for image classification tasks.",
    "Recurrent neural networks process sequential data like text and speech.",
    "Transformers use self-attention mechanisms to process language in parallel.",
    "BERT and GPT are powerful language models built on transformer architecture.",
    "Overfitting occurs when a model learns noise instead of the true pattern.",
    "Regularization techniques like dropout prevent overfitting in deep networks.",
    "Cross-validation helps evaluate how well a machine learning model generalizes.",
    "Feature engineering transforms raw data into meaningful input features.",
    "Support vector machines find the optimal hyperplane to separate classes.",
    "Clustering algorithms like K-means group similar data points together.",
    "Decision trees split data based on feature thresholds to classify samples.",
    "Random forests combine multiple decision trees to improve prediction accuracy.",
]

# ── Build co-occurrence matrix & SVD embeddings ──────────────────────
vectorizer  = CountVectorizer(stop_words='english', min_df=1)
doc_term    = vectorizer.fit_transform(extended_corpus)
vocab       = list(vectorizer.get_feature_names_out())

cooc_matrix = (doc_term.T @ doc_term).toarray().astype(float)
np.fill_diagonal(cooc_matrix, 0)

svd        = TruncatedSVD(n_components=50, random_state=42)
embeddings = normalize(svd.fit_transform(cooc_matrix))  # L2-normalized

print(f"✅ Word embeddings ready!")
print(f"   Vocabulary : {len(vocab)} words")
print(f"   Dimensions : {embeddings.shape[1]}")
print(f"\nSample — vector for 'learning' (first 10 dims):")
print(np.round(embeddings[vocab.index('learning')][:10], 4))

# ── Helper functions ─────────────────────────────────────────────────
def get_vector(word):
    return embeddings[vocab.index(word)] if word in vocab else None

def most_similar(word, topn=5):
    if word not in vocab:
        print(f"  '{word}' not in vocabulary"); return []
    idx      = vocab.index(word)
    sims     = cosine_similarity(embeddings[idx].reshape(1,-1), embeddings).flatten()
    sims[idx] = -1
    top_idx  = np.argsort(sims)[::-1][:topn]
    return [(vocab[i], round(float(sims[i]), 4)) for i in top_idx]

# ── Most similar words ───────────────────────────────────────────────
print("\n📌 Most Similar Words (SVD Co-occurrence Embeddings):\n")
for word in ['learning', 'neural', 'data', 'language']:
    results = most_similar(word, topn=5)
    if results:
        print(f"🔍 '{word}':")
        for sim_word, score in results:
            bar = '█' * int(score * 20)
            print(f"   {sim_word:<18} {score:.4f}  {bar}")
        print()

# ── Word analogies ───────────────────────────────────────────────────
print("🔢 Word Analogies (a - b + c = ?):")
print("-" * 55)
analogies = [
    ('supervised', 'labeled',  'unlabeled'),
    ('deep',       'learning', 'networks'),
    ('neural',     'network',  'learning'),
]
for a, b, c in analogies:
    if all(w in vocab for w in [a, b, c]):
        vec   = embeddings[vocab.index(a)] - embeddings[vocab.index(b)] + embeddings[vocab.index(c)]
        sims  = cosine_similarity(vec.reshape(1,-1), embeddings).flatten()
        for w in [a, b, c]:
            sims[vocab.index(w)] = -1        # exclude input words
        top3  = [(vocab[i], round(float(sims[i]),3)) for i in np.argsort(sims)[::-1][:3]]
        print(f"'{a}' - '{b}' + '{c}' → {top3}")
    else:
        print(f"Skipped — some words not in vocab")

# ── PCA Visualization ────────────────────────────────────────────────
from sklearn.decomposition import PCA

word_groups = {
    'ML Algorithms': ['learning', 'supervised', 'unsupervised', 'classification', 'clustering'],
    'Neural Nets'  : ['neural',   'networks',   'deep',         'layers',         'weights'],
    'NLP'          : ['language', 'text',       'processing',   'transformers',   'models'],
    'Data'         : ['data',     'features',   'training',     'patterns',       'samples'],
}
color_map = {
    'ML Algorithms': '#e74c3c',
    'Neural Nets'  : '#3498db',
    'NLP'          : '#2ecc71',
    'Data'         : '#f39c12'
}

plot_words, plot_labels, plot_colors = [], [], []
for group, words in word_groups.items():
    for word in words:
        if word in vocab:
            plot_words.append(word)
            plot_labels.append(group)
            plot_colors.append(color_map[group])

vectors = np.array([get_vector(w) for w in plot_words])
pca     = PCA(n_components=2, random_state=42)
coords  = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(12, 8))
for group, color in color_map.items():
    mask = [l == group for l in plot_labels]
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        c=color, label=group, s=140, zorder=3,
        edgecolors='white', linewidth=0.8
    )
for i, word in enumerate(plot_words):
    ax.annotate(
        word, (coords[i, 0], coords[i, 1]),
        textcoords='offset points', xytext=(6, 4),
        fontsize=9, fontweight='bold'
    )

ax.set_title("Word Embeddings Visualized in 2D (PCA)", fontsize=14, fontweight='bold')
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('word2vec_pca.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n📌 Semantically related words cluster together in the embedding space")

✅ Word embeddings ready!
   Vocabulary : 155 words
   Dimensions : 50

Sample — vector for 'learning' (first 10 dims):
[ 0.7666 -0.244  -0.0432 -0.1473 -0.0585 -0.     -0.0719 -0.4971  0.0502
 -0.0743]

📌 Most Similar Words (SVD Co-occurrence Embeddings):

🔍 'learning':
   used               0.5210  ██████████
   python             0.5209  ██████████
   popular            0.5203  ██████████
   machine            0.5174  ██████████
   subset             0.5163  ██████████

🔍 'neural':
   uses               0.5969  ███████████
   complex            0.5968  ███████████
   learn              0.5967  ███████████
   layers             0.5962  ███████████
   speech             0.5802  ███████████

🔍 'data':
   machine            0.5560  ███████████
   subset             0.5258  ██████████
   focused            0.5257  ██████████
   patterns           0.5171  ██████████
   uses               0.5023  ██████████

🔍 'language':
   natural            0.5604  ███████████
   enables            0.560

---
## Part 6: Visualizing Word Embeddings with PCA

Word embeddings live in 100-dimensional space — impossible to visualize directly.  
**PCA** (Principal Component Analysis) compresses 100 dims → 2 dims while preserving maximum variance, letting us *see* the semantic clusters.

# Lab is done above

---
## Part 7: Sentence Embeddings with Sentence-BERT

Word2Vec gives vectors per word — but we often need vectors for **whole sentences**. A naive approach is to average word vectors, but this loses sentence structure.

**Sentence-BERT (SBERT)** is a fine-tuned BERT model that produces rich 384-dimensional sentence embeddings using contrastive learning — semantically similar sentences have high cosine similarity.

In [20]:
from sentence_transformers import SentenceTransformer

# Load a lightweight SBERT model (downloads ~90MB)
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Sentence-BERT model loaded!")
print(f"   Model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: 384")
print(f"   Parameters: ~22M (distilled from BERT)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Sentence-BERT model loaded!
   Model: all-MiniLM-L6-v2
   Embedding dimension: 384
   Parameters: ~22M (distilled from BERT)


In [21]:
# Encode all corpus documents into sentence embeddings
sbert_embeddings = sbert_model.encode(corpus, convert_to_numpy=True)

print(f"📊 SBERT Embedding Matrix Shape: {sbert_embeddings.shape}")
print(f"   {len(corpus)} documents × 384 dimensional vectors")

📊 SBERT Embedding Matrix Shape: (10, 384)
   10 documents × 384 dimensional vectors


In [22]:
# ─── Semantic Sentence Similarity ────────────────────────────────
# Compare semantically similar vs dissimilar sentence pairs

sentence_pairs = [
    # Similar pairs
    ("Machine learning trains models from data.",
     "AI systems learn patterns from datasets."),

    ("Neural networks mimic the human brain.",
     "Deep learning models are inspired by neurons."),

    # Dissimilar pairs
    ("Python is used for machine learning.",
     "The weather today is sunny and warm."),

    ("Gradient descent optimizes model weights.",
     "I enjoy eating pizza on weekends."),

    # Tricky — same words, different meaning
    ("The bank approved my loan application.",
     "The river bank was muddy after the rain."),
]

print("📌 Sentence Pair Similarity (SBERT vs TF-IDF):")
print("=" * 90)

for s1, s2 in sentence_pairs:
    # SBERT similarity
    e1    = sbert_model.encode([s1])
    e2    = sbert_model.encode([s2])
    sbert_score = cosine_similarity(e1, e2)[0][0]

    # TF-IDF similarity
    pair_tfidf   = tfidf_vectorizer.transform([s1, s2])
    tfidf_score  = cosine_similarity(pair_tfidf[0], pair_tfidf[1])[0][0]

    print(f"\nSentence 1 : {s1}")
    print(f"Sentence 2 : {s2}")
    print(f"SBERT Sim  : {sbert_score:.4f}  {'🟢 Similar' if sbert_score > 0.5 else '🔴 Dissimilar'}")
    print(f"TF-IDF Sim : {tfidf_score:.4f}  {'🟢 Similar' if tfidf_score > 0.3 else '🔴 Dissimilar'}")

📌 Sentence Pair Similarity (SBERT vs TF-IDF):

Sentence 1 : Machine learning trains models from data.
Sentence 2 : AI systems learn patterns from datasets.
SBERT Sim  : 0.4547  🔴 Dissimilar
TF-IDF Sim : 0.0000  🔴 Dissimilar

Sentence 1 : Neural networks mimic the human brain.
Sentence 2 : Deep learning models are inspired by neurons.
SBERT Sim  : 0.6816  🟢 Similar
TF-IDF Sim : 0.0000  🔴 Dissimilar

Sentence 1 : Python is used for machine learning.
Sentence 2 : The weather today is sunny and warm.
SBERT Sim  : 0.0227  🔴 Dissimilar
TF-IDF Sim : 0.0000  🔴 Dissimilar

Sentence 1 : Gradient descent optimizes model weights.
Sentence 2 : I enjoy eating pizza on weekends.
SBERT Sim  : 0.0078  🔴 Dissimilar
TF-IDF Sim : 0.0000  🔴 Dissimilar

Sentence 1 : The bank approved my loan application.
Sentence 2 : The river bank was muddy after the rain.
SBERT Sim  : 0.2670  🔴 Dissimilar
TF-IDF Sim : 0.0000  🔴 Dissimilar


---
## Part 8: TF-IDF vs SBERT — Head-to-Head Comparison

In [23]:
# Compute cosine similarity matrices for both methods
sbert_sim = cosine_similarity(sbert_embeddings)
tfidf_sim = cosine_similarity(tfidf_matrix)

# Side-by-side heatmap comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, sim_mat, title in zip(
    axes,
    [tfidf_sim, sbert_sim],
    ['TF-IDF Cosine Similarity', 'SBERT Cosine Similarity']
):
    sns.heatmap(
        sim_mat,
        ax=ax,
        xticklabels=doc_labels,
        yticklabels=doc_labels,
        cmap='Blues',
        annot=True,
        fmt='.2f',
        square=True,
        vmin=0, vmax=1,
        linewidths=0.4,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

plt.suptitle("TF-IDF vs SBERT: Document Similarity Comparison",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tfidf_vs_sbert.png', dpi=120, bbox_inches='tight')
plt.show()
print("📌 SBERT captures deeper semantic similarity — notice different scores for the same pairs")

📌 SBERT captures deeper semantic similarity — notice different scores for the same pairs


---
## Part 9: Semantic Search Engine using SBERT

In [24]:
def semantic_search(query, model, embeddings, labels, corpus, top_k=3):
    """
    Semantic search: find documents most similar to the query
    using dense SBERT embeddings instead of keyword matching.
    Handles paraphrases, synonyms, and conceptual similarity.
    """
    query_emb   = model.encode([query])
    scores      = cosine_similarity(query_emb, embeddings).flatten()
    ranked      = np.argsort(scores)[::-1][:top_k]

    print(f"🔍 Query: '{query}'")
    print(f"\n{'Rank':<5} | {'Document':<18} | {'Score':<8} | Content")
    print("-" * 90)
    for rank, idx in enumerate(ranked, 1):
        bar = '█' * int(scores[idx] * 20)
        print(f"{rank:<5} | {labels[idx]:<18} | {scores[idx]:.4f}   | {corpus[idx][:55]}...")
        print(f"       | {'':18} |          | {bar}")


# Test with paraphrase-style queries (would fail with keyword TF-IDF)
semantic_queries = [
    "algorithms that improve from experience",        # → Machine Learning
    "parsing and generating sentences",               # → NLP
    "brain-inspired computing structures",            # → Neural Networks
    "teaching agents via rewards and penalties",      # → Reinforcement Learning
]

for q in semantic_queries:
    print()
    semantic_search(q, sbert_model, sbert_embeddings, doc_labels, corpus)
    print()


🔍 Query: 'algorithms that improve from experience'

Rank  | Document           | Score    | Content
------------------------------------------------------------------------------------------
1     | ML Basics          | 0.3390   | Machine learning is a subset of artificial intelligence...
       |                    |          | ██████
2     | RL                 | 0.3260   | Reinforcement learning trains agents to make decisions ...
       |                    |          | ██████
3     | NLP                | 0.2797   | Natural language processing enables computers to unders...
       |                    |          | █████


🔍 Query: 'parsing and generating sentences'

Rank  | Document           | Score    | Content
------------------------------------------------------------------------------------------
1     | NLP                | 0.4908   | Natural language processing enables computers to unders...
       |                    |          | █████████
2     | TF-IDF             | 0.1

---
## Part 10: Summary Comparison Table

In [25]:
# Final comparison of all vectorization methods
summary = [
    ("Bag of Words",    "Sparse",  "vocab size", "No",  "No",   "Fast",  "Text classification (simple)"),
    ("TF-IDF",          "Sparse",  "vocab size", "No",  "No",   "Fast",  "Search engines, keyword extraction"),
    ("Word2Vec",        "Dense",   "100–300",    "Yes", "No",   "Med",   "Word similarity, word analogies"),
    ("GloVe",           "Dense",   "50–300",     "Yes", "No",   "Med",   "Pretrained embeddings for NLP tasks"),
    ("Sentence-BERT",   "Dense",   "384–768",    "Yes", "Yes",  "Slow",  "Semantic search, sentence similarity"),
]

headers = ["Method", "Type", "Dim", "Semantic?", "Context?", "Speed", "Best For"]

col_widths = [16, 7, 10, 10, 10, 7, 42]
header_str = " | ".join(f"{h:<{w}}" for h, w in zip(headers, col_widths))
print(header_str)
print("-" * (sum(col_widths) + 3 * (len(col_widths) - 1)))
for row in summary:
    row_str = " | ".join(f"{v:<{w}}" for v, w in zip(row, col_widths))
    print(row_str)

print("\n✅ Lab 3 Complete!")
print("\n📌 Rule of thumb:")
print("   • Use TF-IDF for fast, interpretable keyword-based tasks")
print("   • Use SBERT/embeddings when semantic meaning matters")
print("   • For production systems: combine both (hybrid retrieval)")

Method           | Type    | Dim        | Semantic?  | Context?   | Speed   | Best For                                  
------------------------------------------------------------------------------------------------------------------------
Bag of Words     | Sparse  | vocab size | No         | No         | Fast    | Text classification (simple)              
TF-IDF           | Sparse  | vocab size | No         | No         | Fast    | Search engines, keyword extraction        
Word2Vec         | Dense   | 100–300    | Yes        | No         | Med     | Word similarity, word analogies           
GloVe            | Dense   | 50–300     | Yes        | No         | Med     | Pretrained embeddings for NLP tasks       
Sentence-BERT    | Dense   | 384–768    | Yes        | Yes        | Slow    | Semantic search, sentence similarity      

✅ Lab 3 Complete!

📌 Rule of thumb:
   • Use TF-IDF for fast, interpretable keyword-based tasks
   • Use SBERT/embeddings when semantic meaning matters
